In [1]:
import igl
import numpy as np
import open3d as o3d




In [2]:
def read_special_indices_from_obj(obj_file_path):
    # Open the file and read the first line
    with open(obj_file_path, "r") as file:
        first_line = file.readline().strip()

    # Extract the indices from the comment line if it starts with "# Special vertex indices"
    if first_line.startswith("# Coarse Triangle Correspondence"):
        # Remove the prefix and convert the string of indices into a list of integers
        return list(map(int, first_line[len("# Coarse Triangle Correspondence: "):].split(", ")))

    print(f"No special vertex indices found in {obj_file_path}")
    return None

In [3]:
# Load mesh with Open3D

file_name = "patches/sphere2_20patch/0.obj"



def tutte_embedding(input_filename, output_filename):
    
    mesh = o3d.io.read_triangle_mesh(input_filename)
    coarse_triangle_vtx_indices = read_special_indices_from_obj(input_filename)
    
    V = np.asarray(mesh.vertices)  # Vertices as a numpy array
    F = np.asarray(mesh.triangles)  # Faces as a numpy array
    
    # Get boundary loop
    boundary = list ( igl.boundary_loop(F) )
    
    print('boundary:', boundary)
    print(coarse_triangle_vtx_indices)
    
    
    ###########################################################################
    '''
    # Map boundary vertices to the unit circle (convex polygon)
    num_boundary = len(boundary)
    angles = np.linspace(0, 2 * np.pi, num_boundary, endpoint=False)
    boundary_uv = np.column_stack((np.cos(angles), np.sin(angles)))
    '''
    
    # Triangle corners: unit side length, one at origin
    triangle_corners = np.array([
        [0.0, 0.0],
        [1.0, 0.0],
        [0.5, np.sqrt(3)/2]
    ])
    
    # Find boundary indices of the 3 special vertices in order
    corner_idx = [boundary.index(idx) for idx in coarse_triangle_vtx_indices]
    
    # Ensure corner indices are in order
    corner_idx = sorted(corner_idx)
    
    # Break the boundary into 3 segments between corners (looped)
    segments = [
        boundary[corner_idx[0]:corner_idx[1]+1],
        boundary[corner_idx[1]:corner_idx[2]+1],
        boundary[corner_idx[2]:] + boundary[:corner_idx[0]+1]
    ]
    
    # Total boundary length
    bdry_points = np.array([V[idx] for idx in boundary])
    lengths = np.linalg.norm(np.roll(bdry_points, -1, axis=0) - bdry_points, axis=1)
    total_length = np.sum(lengths)
    
    # Map each segment to a triangle edge proportionally
    boundary_uv = np.zeros((len(boundary), 2))
    
    for seg, (start_pt, end_pt) in zip(segments, zip(triangle_corners, np.roll(triangle_corners, -1, axis=0))):
        # Compute cumulative arclength along this segment
        seg_points = np.array([V[i] for i in seg])
        seg_lengths = np.linalg.norm(np.diff(seg_points, axis=0), axis=1)
        seg_lengths = np.insert(np.cumsum(seg_lengths), 0, 0.0)
        seg_total = seg_lengths[-1]
    
        # Linearly interpolate UVs along this edge
        for i, s in enumerate(seg):
            t = seg_lengths[i] / seg_total if seg_total > 0 else 0
            uv = (1 - t) * start_pt + t * end_pt
            boundary_uv[boundary.index(s)] = uv
    
            
    
    
    ###########################################################################
    
    
    
    # Indices of boundary vertices and their target UVs
    b = np.array(boundary, dtype=np.int32)
    bc = boundary_uv.astype(np.float64)
    
    # Harmonic mapping (Tutte-style, k=1)
    uv = igl.harmonic(V, F, b, bc, 1)
    
    
    with open(output_filename, 'w') as f:
        # Write vertices
        for vertex in V:
            f.write(f"v {vertex[0]} {vertex[1]} {vertex[2]}\n")
        
        # Write texture coordinates (UVs)
        for uv_coord in uv:
            f.write(f"vt {uv_coord[0]} {uv_coord[1]}\n")
    
        # Write faces (adjust to include texture coordinates)
        for face in F:
            # OBJ format requires faces to include texture coordinates: 
            # "f v1/vt1 v2/vt2 v3/vt3"
            f.write(f"f {face[0]+1}/{face[0]+1} {face[1]+1}/{face[1]+1} {face[2]+1}/{face[2]+1}\n")
    
    print(f"Tutte-style harmonic embedding completed. UVs saved to {output_filename}")


In [4]:


# Example inputs
# V: (N, 3) array of 3D vertex positions
# F: (M, 3) array of triangle indices
# UV: (N, 2) array of UV coordinates
# boundary: list of boundary indices (optional)
# coarse_triangle_vtx_indices: any extra data you want to print

# ---- Replace these with actual data ----
# V = np.array(...)       # (N, 3)
# F = np.array(...)       # (M, 3)
# UV = np.array(...)      # (N, 2)
# boundary = [...]
# coarse_triangle_vtx_indices = [...]






NameError: name 'uv' is not defined